-- Load 2021-2026 and build bt

In [1]:
import nflreadpy as nfl
import pandas as pd
from sklearn.linear_model import LinearRegression

stats_all = nfl.load_player_stats(list(range(2021, 2027))).to_pandas()
wr_hist = stats_all[(stats_all["position"] == "WR") & (stats_all["season_type"] == "REG")].copy()
wr_hist["fp"] = wr_hist["fantasy_points"]          # standard scoring

bt = wr_hist.sort_values(["player_id", "season", "week"]).copy()
bt["t"] = bt["season"] * 100 + bt["week"]

g = bt.groupby("player_id")
for col in ["fp", "targets", "wopr"]:
    bt[col + "_r8"] = g[col].transform(lambda s: s.shift(1).rolling(8, min_periods=4).mean())

bt = bt.dropna(subset=["fp", "fp_r8", "targets_r8", "wopr_r8"])
bt = bt[bt["targets_r8"] >= 4]
print(len(bt), "player-games")

5693 player-games


Walk forward, building res

In [2]:
parts = []
for t in sorted(bt["t"].unique()):
    train = bt[bt["t"] < t]
    test = bt[bt["t"] == t].copy()
    if len(train) < 500 or test.empty:
        continue
    m = LinearRegression().fit(train[["targets_r8", "wopr_r8"]], train["fp_r8"])
    test["exp_r8"] = m.predict(test[["targets_r8", "wopr_r8"]])
    parts.append(test)

res = pd.concat(parts)
res["gap_r8"] = res["fp_r8"] - res["exp_r8"]
print(len(res), "tested player-games across", res["season"].nunique(), "seasons")

5181 tested player-games across 6 seasons


Star-player test

In [3]:
res = res.sort_values(["player_id", "t"])
res["own_gap"] = res.groupby("player_id")["gap_r8"].transform(
    lambda s: s.shift(8).expanding(min_periods=1).mean()
)
sub = res.dropna(subset=["own_gap"]).copy()
sub["gap_adj"] = sub["gap_r8"] - sub["own_gap"]
print(len(sub), "player-games with a long enough history")

rows = []
for lam in [0, 0.25, 0.5, 0.75, 1.0]:
    rows.append({
        "lam": lam,
        "plain": (sub["fp"] - (sub["fp_r8"] - lam * sub["gap_r8"])).abs().mean(),
        "player_adj": (sub["fp"] - (sub["fp_r8"] - lam * sub["gap_adj"])).abs().mean(),
    })
pd.DataFrame(rows).round(3)

3813 player-games with a long enough history


,lam,plain,player_adj
0,0.00,4.784,4.784
1,0.25,4.747,4.757
2,0.50,4.741,4.766
3,0.75,4.762,4.809
4,1.00,4.813,4.881


Results by season

In [4]:
out = {}
for lam in [0, 0.25, 0.5, 0.75]:
    err = (res["fp"] - (res["fp_r8"] - lam * res["gap_r8"])).abs()
    out[f"lam={lam}"] = err.groupby(res["season"]).mean()
pd.DataFrame(out).round(3)

,lam=0,lam=0.25,lam=0.5,lam=0.75
season,,,,
2021,4.444,4.450,4.512,4.605
2022,4.632,4.579,4.556,4.565
2023,4.735,4.715,4.726,4.767
2024,4.782,4.723,4.694,4.688
2025,4.652,4.618,4.612,4.627
2026,4.827,4.774,4.750,4.752


Decile table

In [5]:
res["change"] = res["fp"] - res["fp_r8"]
res["decile"] = res.groupby("t")["gap_r8"].transform(
    lambda s: pd.qcut(s, 10, labels=False, duplicates="drop")
)
d = res.groupby("decile")["change"].agg(["mean", "count", "std"])
d["sem"] = d["std"] / d["count"] ** 0.5
d.round(2)

,mean,count,std,sem
decile,,,,
0,1.15,554,5.78,0.25
1,1.05,517,5.58,0.25
2,0.62,509,5.40,0.24
3,-0.16,511,5.06,0.22
4,-0.06,524,5.97,0.26
5,-0.31,497,5.77,0.26
6,-1.50,501,5.59,0.25
7,-0.71,519,6.29,0.28
8,-1.48,507,6.59,0.29


Quartile check

In [6]:
res["fp_bin"] = pd.qcut(res["fp_r8"], 4, labels=["low", "mid-low", "mid-high", "high"])
res["gap_high"] = res["gap_r8"] > res.groupby("fp_bin", observed=True)["gap_r8"].transform("median")

(res.groupby(["fp_bin", "gap_high"], observed=True)["change"]
    .agg(["mean", "count"]).round(2))

mean  count
fp_bin   gap_high             
low      False     1.31    651
         True     -0.01    650
mid-low  False     1.05    648
         True     -0.49    648
mid-high False    -0.05    645
         True     -1.22    644
high     False    -1.42    648
         True     -2.41    647

Direct vs blend comparison

In [7]:
parts = []
for t in sorted(bt["t"].unique()):
    train = bt[bt["t"] < t]
    test = bt[bt["t"] == t].copy()
    if len(train) < 500 or test.empty:
        continue
    u = LinearRegression().fit(train[["targets_r8", "wopr_r8"]], train["fp_r8"])
    test["gap_r8"] = test["fp_r8"] - u.predict(test[["targets_r8", "wopr_r8"]])
    X = ["fp_r8", "targets_r8", "wopr_r8"]
    d = LinearRegression().fit(train[X], train["fp"])
    test["pred_direct"] = d.predict(test[X])
    parts.append(test)

cmp = pd.concat(parts)
for name, pred in [
    ("naive (8-game avg)", cmp["fp_r8"]),
    ("blend lam=0.5", cmp["fp_r8"] - 0.5 * cmp["gap_r8"]),
    ("direct regression", cmp["pred_direct"]),
]:
    print(f"{name:<20} MAE {(cmp['fp'] - pred).abs().mean():.3f}")

naive (8-game avg)   MAE 4.690
blend lam=0.5        MAE 4.643
direct regression    MAE 4.560


In [8]:
X = ["fp_r8", "targets_r8", "wopr_r8"]
final = LinearRegression().fit(bt[X], bt["fp"])
tier = LinearRegression().fit(bt[["fp_r8"]], bt["fp"])
print("Coefficients:", dict(zip(X, final.coef_.round(3))), "intercept:", round(final.intercept_, 2))

latest = wr_hist.sort_values(["player_id", "season", "week"]).groupby("player_id").tail(8)

cur = (
    latest.groupby(["player_id", "player_display_name"])
          .agg(games=("week", "count"),
               games_2026=("season", lambda s: (s == 2026).sum()),
               fp_r8=("fp", "mean"),
               targets_r8=("targets", "mean"),
               wopr_r8=("wopr", "mean"))
          .reset_index()
)
cur = cur[(cur["games"] >= 4) & (cur["games_2026"] > 0) & (cur["targets_r8"] >= 4)].dropna()

cur["projection"] = final.predict(cur[X])
cur["tier_proj"] = tier.predict(cur[["fp_r8"]])
cur["usage_effect"] = cur["projection"] - cur["tier_proj"]

lo, hi = cur["usage_effect"].quantile([0.10, 0.90])
cur["flag"] = "hold"
cur.loc[cur["usage_effect"] >= hi, "flag"] = "buy-low"
cur.loc[cur["usage_effect"] <= lo, "flag"] = "sell-high"
cur["note"] = cur["games_2026"].apply(lambda g: "mostly 2025 data" if g <= 2 else "")

cols = ["player_display_name", "flag", "fp_r8", "projection", "usage_effect", "games_2026", "note"]
cur.sort_values("projection", ascending=False)[cols].round(2).head(30)

Coefficients: {'fp_r8': np.float64(0.419), 'targets_r8': np.float64(0.216), 'wopr_r8': np.float64(4.565)} intercept: 0.25


,player_display_name,flag,fp_r8,projection,usage_effect,games_2026,note
332,Jaxon Smith-Njigba,hold,17.58,14.25,0.51,3,
373,Puka Nacua,hold,17.76,13.26,-0.60,2,mostly 2025 data
268,Amon-Ra St. Brown,hold,14.56,12.53,0.78,3,
284,Chris Olave,hold,14.55,12.18,0.43,3,
371,Zay Flowers,hold,15.45,11.81,-0.53,3,
257,Ja'Marr Chase,hold,12.11,10.20,0.06,4,
218,CeeDee Lamb,hold,12.05,10.17,0.07,4,
225,Tee Higgins,hold,12.15,10.08,-0.09,4,
334,Michael Wilson,buy-low,10.65,10.01,0.83,4,
237,Nico Collins,hold,13.02,9.91,-0.84,2,mostly 2025 data


In [9]:
X = ["fp_r8", "targets_r8", "wopr_r8"]
parts = []
for t in sorted(bt["t"].unique()):
    train = bt[bt["t"] < t]
    test = bt[bt["t"] == t].copy()
    if len(train) < 500 or test.empty:
        continue
    full = LinearRegression().fit(train[X], train["fp"])
    tier_m = LinearRegression().fit(train[["fp_r8"]], train["fp"])
    test["usage_effect"] = full.predict(test[X]) - tier_m.predict(test[["fp_r8"]])
    test["beat_tier"] = test["fp"] - tier_m.predict(test[["fp_r8"]])
    parts.append(test)

v = pd.concat(parts)
v["decile"] = v.groupby("t")["usage_effect"].transform(
    lambda s: pd.qcut(s, 10, labels=False, duplicates="drop")
)
d = v.groupby("decile")["beat_tier"].agg(["mean", "count", "std"])
d["sem"] = d["std"] / d["count"] ** 0.5
d.round(2)

,mean,count,std,sem
decile,,,,
0,-1.37,554,5.71,0.24
1,-0.53,517,5.60,0.25
2,-0.37,509,5.52,0.24
3,-0.53,511,5.26,0.23
4,-0.07,524,5.45,0.24
5,0.63,497,5.97,0.27
6,0.23,501,6.06,0.27
7,0.64,519,6.04,0.26
8,0.75,507,6.56,0.29
